In [0]:
%pip install -U mlflow scikit-learn skops
dbutils.library.restartPython()

In [0]:
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from pyspark.sql import functions as F
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

dbutils.widgets.text("catalog", "ieee_cis")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

MODEL_NAME = f"{catalog}.ml.fraud_model"
user = spark.sql("SELECT current_user()").first()[0]

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{user}/fraud-detection")
client = MlflowClient()

## 1. Load the challenger and its validation rows

In [0]:
mv = client.get_model_version_by_alias(MODEL_NAME, "challenger")
valid_rows = int(client.get_run(mv.run_id).data.params["valid_rows"])   # params come back as strings

# sklearn flavour, not pyfunc: we need predict_proba, not 0/1 predictions
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{mv.version}")
features = list(model.feature_names_in_) # the exact columns this version was trained on

print(f"challenger = v{mv.version}  run = {mv.run_id}  features = {len(features)}")

In [0]:
# The exact rows 02_train held out for this run
cols = list(dict.fromkeys(["TransactionID", "event_ts", "TransactionAmt", "isFraud", *features]))
valid_pdf = (spark.table("ml.split_sets")
             .where((F.col("run_id") == mv.run_id) & (F.col("split") == "valid"))
             .select(*cols)
             .toPandas()
             .sort_values("TransactionID")
             .reset_index(drop=True))

assert len(valid_pdf) == valid_rows, (
    f"ml.split_sets has {len(valid_pdf):,} validation rows for run {mv.run_id}, 02_train logged {valid_rows:,}. "
    "Re-run 02_train if this version was trained before it saved its validation rows.")

X_valid, y_valid = valid_pdf[features], valid_pdf["isFraud"].astype(int)
print(f"valid = {len(valid_pdf):,} rows   fraud rate = {y_valid.mean():.4f}")

## 2. Score and measure


In [0]:
proba = model.predict_proba(X_valid)[:, 1]               # column 1 = probability of fraud
auprc = average_precision_score(y_valid, proba)
roc_auc = roc_auc_score(y_valid, proba)

# Pick the threshold with the best F1.
# thresholds has one fewer entry than precision/recall, so drop their last value first.
precision, recall, thresholds = precision_recall_curve(y_valid, proba)
p, r = precision[:-1], recall[:-1]
f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
best = int(np.argmax(f1))
threshold = float(thresholds[best])

flagged = (proba >= threshold).astype(int)
prec = precision_score(y_valid, flagged, zero_division=0)
rec = recall_score(y_valid, flagged, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_valid, flagged, labels=[0, 1]).ravel()

print(f"AUPRC     = {auprc:.4f}   (random guessing ≈ {y_valid.mean():.4f})")
print(f"ROC AUC   = {roc_auc:.4f}   (random guessing = 0.5)")
print(f"threshold = {threshold:.4f}   precision = {prec:.3f}   recall = {rec:.3f}   F1 = {f1[best]:.3f}")
print(f"Of {tp + fn:,} frauds in the hidden weeks, the model caught {tp:,}; "
      f"{fp:,} of its {tp + fp:,} flags were false alarms.")

## 3. Save the results


In [0]:
# 02_train already logged val_auprc / val_roc_auc on the run; add the threshold-based metrics next to them.
with mlflow.start_run(run_id=mv.run_id):
    mlflow.log_metrics({"val_precision": prec, "val_recall": rec,
                        "val_f1": float(f1[best]), "val_threshold": threshold})

# The threshold travels with the model version; 04_score reads it from here.
client.set_model_version_tag(MODEL_NAME, mv.version, "threshold", str(threshold))
client.set_model_version_tag(MODEL_NAME, mv.version, "auprc", str(auprc))

In [0]:
# gold.model_metrics: one row per version. Delete this version's row first so a re-run doesn't duplicate it.
metrics_pdf = pd.DataFrame([{
    "version": int(mv.version), "auprc": float(auprc), "roc_auc": float(roc_auc),
    "precision": float(prec), "recall": float(rec), "threshold": threshold,
    "caught": int(tp), "missed": int(fn), "false_alarms": int(fp),
}])
if spark.catalog.tableExists("gold.model_metrics"):
    spark.sql(f"DELETE FROM gold.model_metrics WHERE version = {int(mv.version)}")
(spark.createDataFrame(metrics_pdf)
 .withColumn("evaluated_at", F.current_timestamp())
 .write.mode("append").saveAsTable("gold.model_metrics"))

# gold.validation_predictions: the latest evaluation only
preds_pdf = valid_pdf[["TransactionID", "event_ts", "TransactionAmt"]].assign(
    actual=y_valid.values, fraud_score=proba, flagged=flagged, model_version=int(mv.version))
(spark.createDataFrame(preds_pdf)
 .write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("gold.validation_predictions"))

display(spark.table("gold.model_metrics").orderBy("version"))

## 4. Champion / challenger: promote only if better

In [0]:
try:
    champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
except MlflowException:                                  # no champion alias yet
    champion = None

if champion is None:
    promote, reason = True, "no champion yet"
elif champion.version == mv.version:
    promote, reason = True, f"v{mv.version} is already the champion"
else:
    champion_auprc = float(champion.tags["auprc"])
    promote = auprc > champion_auprc
    reason = (f"challenger v{mv.version} AUPRC {auprc:.4f} vs "
              f"champion v{champion.version} AUPRC {champion_auprc:.4f}")

if promote:
    client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
client.set_model_version_tag(MODEL_NAME, mv.version, "promoted", str(promote).lower())

print(("Promoted: " if promote else "Kept the current champion: ") + reason)